In [ ]:
import sys

helpers_folder = "./helpers"
sys.path.append(helpers_folder)

from plotting_functions import *
from data_processing import *

In [ ]:
with open("config/pathways_variable_list.yml", "r") as varfile:
    all_variables = yaml.safe_load(varfile)

VAR_MAPPING = generate_variable_mapping_from_list(all_variables, biomass_allocation=True)
VAR_MAPPING["sector - subsector"] = VAR_MAPPING[['sector', 'subsector']].agg(' - '.join, axis=1)
VAR_MAPPING["sector - fuel"] = VAR_MAPPING[['sector', 'fuel']].agg(' - '.join, axis=1)
var2sector = VAR_MAPPING.set_index("variable")["sector"].to_dict()
var2fuel = VAR_MAPPING.set_index("variable")["fuel"].to_dict()

In [ ]:



COLORMAPS = {}
for col in VAR_MAPPING:
    variables = list(VAR_MAPPING[col].unique())
    newdict = {}
    for sector, bcolor in basecolors.items():
        subset_variables = [v for v in variables if v.startswith(sector)]
        colors = gradient_from_base(bcolor, len(subset_variables))
        for v, c in zip(subset_variables, colors):
            newdict[v] = c
    COLORMAPS[col] = newdict
COLORMAPS["midpoint"] = midpoint2hex
COLORMAPS["midpoint (excl. CC)"] = midpoint2hex
COLORMAPS["sector"] = basecolors




In [ ]:
def get_default_label(fp):
    premise_run = fp.split("/")[-2]

    return premise_run.split("_")[0]

In [ ]:
output_folder = "output"
baseyear = 2025
expected_no_of_result_files = 53
use_adapted_pm = True
use_adapted_cc = True

## Plot 1: 2050 endpoint comparison with main scenarios

In [ ]:
labels = [
    "NPi",
    "1.5deg_base",
    "1.5deg_intervs",
]
data = get_plotting_data(output_folder, labels, VAR_MAPPING, resultsfolder="main", expected_no_of_result_files=expected_no_of_result_files,
                         use_adapted_pm=use_adapted_pm, use_adapted_cc=use_adapted_cc)
errdata = get_plotting_data(output_folder, labels, VAR_MAPPING, resultsfolder="MC", expected_no_of_result_files=expected_no_of_result_files,
                         datacols=["year", "impact_category", "sample index"],
                         use_adapted_pm=use_adapted_pm, use_adapted_cc=use_adapted_cc)
errdata_harmonized = harmonize_errordata(data, errdata)

### Data preparation

In [ ]:
endpoints = ["ecosystem quality no LT", "ecosystem quality", "human health no LT", "human health", "natural resources"]
mgroups = get_endpoint_method_groups(data[labels[0]], endpoints)

scen_year = [
    f"NPi - {baseyear}",
    "NPi - 2050",
    "1.5deg_base - 2050",
    "1.5deg_intervs - 2050",
]

columns_data = get_columnsdata(data, scen_year)
columns_errdata = get_columnsdata(errdata_harmonized, scen_year)

In [ ]:
ticklabels = [f"{str(baseyear)}", "CP", "NZ", "NZ-SCI"]
endpoints_plot = ["human health", "ecosystem quality"]
p = 3.6
detail_midpoints = ["particulate matter formation",
                    "human toxicity", "land use"]
endpoint_scaling = {"human health": 1e-06, "ecosystem quality": 1e-03}
ENDPOINT2UNIT.update({"human health": "mil. DALY", "ecosystem quality": "species loss (thousands)"})
MIDPOINT2UNIT.update(
    {"particulate matter formation": "mil. DALY", "human toxicity": "mil. DALY", "land use": "species loss (thousands)"})
midpoint_scaling = {"particulate matter formation": 1e-06, "human toxicity": 1e-06, "land use": 1e-03}


fig = plt.figure(layout="constrained", figsize=(12, 8))
subfigs = fig.subfigures(1, 2, wspace=0, width_ratios=[0.3, 0.7])

ep_shares = endpoint_stackplots(columns_data, subfigs[0], endpoints_plot,
                    mgroups, scen_year, ticklabels, scalings=endpoint_scaling,
                    errdata=columns_errdata, tickrotation=0, return_shares=True)
mp_shares = midpoint_stackplots(columns_data, subfigs[1], detail_midpoints,
                    scen_year, ticklabels, errdata=columns_errdata, 
                    scalings=midpoint_scaling, p=p, legend_y_offset=-0.01,
                    tickrotation=0, return_shares=True,
                    )
label_axes(fig.get_axes())
# fig.savefig("plot01_endpoint_overview_v2.pdf")


Shares of midpoints:

In [ ]:
# climate change
for ep in ep_shares:
    cc_shares = ep_shares[ep]["climate change"]
    print(f"CC-Shares for {ep}:\n{cc_shares}\n")

In [ ]:
# NZ shares
for ep in ep_shares:
    nz_shares = ep_shares[ep].loc[scen_year[2]]
    nz_shares_noCC = nz_shares.drop("climate change")
    nz_shares_noCC = nz_shares_noCC.div(nz_shares_noCC.sum())
    print(f"NZ-Shares for {ep}:\n{nz_shares_noCC}\n")

## Reductions relative to 2020

In [ ]:
mp_shares["particulate matter formation"]["act_category"]

roadelec = mp_shares["particulate matter formation"]["act_category"][["road transport", "electricity production"]]

In [ ]:
mp_shares["particulate matter formation"]["act_category"]["industrial heat"]

In [ ]:
mp_shares["particulate matter formation"]["act_category"].loc["1.5deg_base - 2050"]

In [ ]:
roadelec.sum(axis=1)

## Stem plots

In [ ]:
endpoints = ["human health", "ecosystem quality", "natural resources"]
height_ratios=[0.2, 0.8]
labels = ["CP", "NZ", "NZ-SCI"]
colors = ["gray", "C0", "C2"]

fig, axs = plt.subplots(2, 1, sharex=True, height_ratios=height_ratios, figsize=(6, 8))

dflist = []
for ep in endpoints:
    mgroup = mgroups[ep]
    sel = columns_data[columns_data["impact_category"].isin(mgroup)]
    df = sel.groupby(["scenario-year", "midpoint"])["value"].sum().reset_index()
    df["endpoint"] = ep
    df["midpoint-endpoint"] = df["midpoint"] + " (" + df["endpoint"] + ")"
    dflist.append(df)
sdata = pd.concat(dflist).set_index(["scenario-year"])

dflist = []
for ep in endpoints:
    mgroup = mgroups[ep]
    sel = columns_errdata[columns_errdata["impact_category"].isin(mgroup)]
    df = sel.groupby(["scenario-year", "midpoint", "sample index"])["value"].sum().reset_index()
    df["endpoint"] = ep
    df["midpoint-endpoint"] = df["midpoint"] + " (" + df["endpoint"] + ")"
    dflist.append(df)
sdata_err = pd.concat(dflist).set_index("scenario-year")

ax = axs[0]
make_stemplot(sdata, ax, scen_year, errdata=sdata_err, idx="endpoint", labels=labels, 
              sortlabel="NZ", colors=colors,
              plot_reduction=True, reduction_base="CP", reduction_target="NZ", reduction_offset=0.3,
              color_by_endpoints="box", logscale=True)
ax.margins(y=0.4)
handles = []
for lbl, c in zip(labels, colors):
    handles.append(Patch(color=c, label=lbl))
ax.legend(handles=handles, loc="upper right")
ax.set_title("Endpoint")

ax = axs[1]
make_stemplot(sdata, ax, scen_year, errdata=sdata_err, idx="midpoint", labels=labels, sortlabel="NZ",
              plot_reduction=True, reduction_base="CP", reduction_target="NZ", reduction_offset=0.3,
              color_by_endpoints="bracket", logscale=True, bracket_offset=0.35, patch_width=0.05, alpha=0.5)
ax.set_title("Midpoint")
ax.set_yticks(ax.get_yticks(), [textwrap.fill(t.get_text(), 20) for t in ax.get_yticklabels()])
ax.set_xticks([0.1, 0.2, 0.5, 1, 2, 5])


fig.supxlabel(f"Impacts in 2050, relative to {baseyear}")

ax_annotate = axs[1]
# ax_annotate.annotate("", xy=(1.5, 12.5), xytext=(1.51, 12.5), va="center", clip_on=False,
#                     arrowprops=dict(arrowstyle='-[, widthB=2.5, lengthB=1', color="black", lw=0.8, alpha=0.7))
# ax_annotate.annotate("", xy=(1.5, 8), xytext=(1.51, 8), va="center",
#                     arrowprops=dict(arrowstyle='-[, widthB=5.5, lengthB=1', color="black", lw=0.8, alpha=0.7))
# ax_annotate.annotate("", xy=(1.51, 12.5), xytext=(2.5, 10.1), va="center", ha="left",
#                     arrowprops=dict(arrowstyle='-', color="black", lw=0.8, alpha=0.7))
# ax_annotate.annotate("", xy=(1.51, 8), xytext=(2.5, 9.9), va="center", ha="left",
#                     arrowprops=dict(arrowstyle='-', color="black", lw=0.8, alpha=0.7))
# ax_annotate.text(3, 10, "(co)benefits", va="center", ha="left", clip_on=False)
ax_annotate.annotate("(Co-)benefits", xy=(1.5, 10), xytext=(2.5, 10), va="center", clip_on=False,
                    arrowprops=dict(arrowstyle='-[, widthB=8.5, lengthB=1', color="black", lw=0.8, alpha=0.7))
ax_annotate.annotate("Tradeoffs", xy=(0.45, 3.5), xytext=(0.2, 3.5), va="center", ha="right",clip_on=False,
                    arrowprops=dict(arrowstyle='-[, widthB=6.5, lengthB=1', color="black", lw=0.8, alpha=0.7))
    
label_axes(fig.get_axes(), position=[-0.05, 1.05])

In [ ]:
1- get_stemdata(sdata, "endpoint", scen_year, labels, sortlabel="NZ")

In [ ]:
(get_stemdata(sdata, "midpoint", scen_year, labels, sortlabel="NZ") - 1) * 100

In [ ]:
materials = get_stemdata(sdata, "midpoint", scen_year, labels, sortlabel="NZ").loc["material resources"]
materials = materials.div(materials["CP"], axis=0)
materials

## Individual interventions

In [ ]:
forward_sequence = [
    "1.5deg_base",
    "1stgenlimit",
    "lowbio",
    "lowbio+slag",
    "lowbio+waste",
    "lowbio+waste+copper",
    "lowbio+waste+copper+eff",
    "lowbio+waste+recycling",
    "lowbio+waste+recycling+shipping",
    "lowbio+waste+recycling+shipping+woodstoves",
    "1.5deg_intervs",
]

backward_sequence = [
    "1.5deg_base",
    "smelting",
    "smelting+woodstoves",
    "APcontrol",
    "APcontrol+otherrecycling",
    "APcontrol+otherrecycling+eff",
    "APcontrol+recycling",
    "APcontrol+recycling+tailings",
    "APcontrol+recycling+waste",
    "APcontrol+recycling+waste+purposelimit",
    "1.5deg_intervs",
]

interventionnames = [
    "1st generation\nphaseout",
    "energy crop\nlimit",
    "slags\ntreatment",
    "tailings\ntreatment",
    "copper\nrecycling",
    "metal\nefficiency",
    "other\nrecycling",
    "shipping",
    "woodstoves",
    "smelting",
]

In [ ]:
broad_indices, bundles = build_water_indices(
    forward_sequence,
    backward_sequence,
    interventionnames,
    {
        "biomass\nconstraints": 2,
        "waste\ntreatment": 2,
        "recycling and\nefficiency": 3,
        "air pollution\ncontrol": 3,
    },
    startname="NZ",
    endname="NZ-SCI",
)

In [ ]:
needed_scens = set(forward_sequence + backward_sequence)

In [ ]:
data2 = get_plotting_data(output_folder, needed_scens, VAR_MAPPING, resultsfolder="main", include_level_in_label=False,
                          replace_in_label={"SSP2-PkBudg750-": ""},
                          datacols=["act_category", "year", "impact_category"],
                          select_cols={"year": 2050}, expected_no_of_result_files=expected_no_of_result_files,
                         use_adapted_pm=use_adapted_pm, use_adapted_cc=use_adapted_cc)

In [ ]:
methods = mgroups["human health"] + mgroups["ecosystem quality"]
rdata = get_reductiondata(data2, methods, index=["scenario", "endpoint"], columns=["midpoint"])
rdata_noCC = rdata.drop(columns="climate change")

In [ ]:
reductions_waterfalls_new(
    rdata_noCC,
    broad_indices,
    bundles,
    midpoint2hex_reordered,
    ["human health", "ecosystem quality"]
)

fig = plt.gcf()
label_axes(fig.get_axes(), position=[-0.05, 1.05])

### Fw and bw only (for SI)

In [ ]:
def reverse_dict(d):
    return dict(reversed(list(d.items())))

In [ ]:
def get_ith_step(windex, i, reverse_steps=False):
    newindex = {}
    newindex["start"] = windex["start"]
    newindex["end"] = windex["end"]
    steps = {k: v[i:i+1] for k, v in windex["steps"].items()}
    if reverse_steps:
        steps = reverse_dict(steps)
    newindex["steps"] = steps

    return newindex

In [ ]:
broad_indices_fw = get_ith_step(broad_indices, 0)
broad_indices_bw = get_ith_step(broad_indices, 1, reverse_steps=True)
bundles_fw = {k: get_ith_step(v, 0) for k, v in bundles.items()}
bundles_bw = {k: get_ith_step(v, 1, reverse_steps=True) for k, v in bundles.items()}
bundles_bw = reverse_dict(bundles_bw)

In [ ]:
reductions_waterfalls_new(
    rdata_noCC,
    broad_indices_fw,
    bundles_fw,
    midpoint2hex_reordered,
    ["human health", "ecosystem quality"]
)

fig = plt.gcf()
label_axes(fig.get_axes(), position=[-0.05, 1.05])

In [ ]:
reductions_waterfalls_new(
    rdata_noCC,
    broad_indices_bw,
    bundles_bw,
    midpoint2hex_reordered,
    ["human health", "ecosystem quality"]
)

fig = plt.gcf()
label_axes(fig.get_axes(), position=[-0.05, 1.05])